# Frozen-encoder ranking backends

Tests whether the recurrent-author problem comes from cosine geometry or candidate popularity. The encoder and source-heldout split stay frozen. Transform fitting uses train only; hyperparameters use dev only; test is read once for the final comparison.

Backends: cosine, centered cosine, all-but-top, shrinkage whitening, L1, Spearman, CSLS, adaptive S-Norm, regularized PLDA, and PLDA + S-Norm. Normalized Euclidean is omitted because it induces exactly the same ranking as cosine.

## Evidence and decision rule

Whitening addresses anisotropic sentence spaces (Su et al., 2021). CSLS and local scaling target hubs rather than manually penalizing named authors (Conneau et al., 2017; Schnitzer et al., 2012). S-Norm and PLDA are cohort-normalized and probabilistic scoring alternatives from speaker verification. Recent retrieval papers motivate, but do not establish, the 2026 hubness hypotheses for stylometry.

A backend passes only if its paired-profile MRR confidence interval is no worse than −0.01, Recall@3 falls by less than 0.01, and both false-top3 HHI and Gini improve over cosine. No passing backend means no production ranking change.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib.util, json, os, subprocess, sys
missing = [name for name in ('pyarrow', 'sklearn', 'sentence_transformers') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow>=14', 'scikit-learn>=1.4', 'sentence-transformers>=4.1'], check=True)
import pandas as pd

REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
GUTENBERG = REPO / 'artifacts/source_expansion_v2/gutenberg_targeted_v1'
HELDOUT = GUTENBERG / 'source_heldout_splits.parquet'
MODEL = REPO / 'artifacts/multilingual_author_style_v1'
EMBEDDINGS = GUTENBERG / 'frozen_encoder_eval'
OUT = GUTENBERG / 'similarity_backends_v1'
SEED = 20260902
assert HELDOUT.exists(), HELDOUT
assert (MODEL / 'model.safetensors').exists(), MODEL

def run(command):
    print('>>>', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {command}')


In [ ]:
# Encodes once. Reruns reuse aligned train/dev/test arrays.
run([
    sys.executable, 'scripts/style_embedding_recall.py',
    '--input', str(HELDOUT),
    '--out-dir', str(EMBEDDINGS),
    '--model-name', str(MODEL),
    '--batch-size', '128', '--train-cap', '300',
    '--eval-splits', 'dev,test', '--device', 'cuda',
    '--seed', str(SEED), '--skip-existing',
])

In [ ]:
# CPU matrix work after the embedding cell; GPU is not expected to stay busy.
run([
    sys.executable, 'scripts/evaluate_similarity_backends.py',
    '--input', str(HELDOUT),
    '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(OUT),
    '--train-cap', '300', '--bootstrap-runs', '5000',
    '--seed', str(SEED),
])

In [ ]:
metrics = pd.read_csv(OUT / 'backend_test_metrics.csv')
display(metrics[[
    'method', 'selected_parameter', 'mrr', 'mrr_ci_low', 'mrr_ci_high',
    'recall_at_1', 'recall_at_3', 'recall_at_5',
    'false_top3_hhi', 'false_top3_gini',
    'worst_decile_profile_recall_at_3', 'subgroup_non_degradation', 'adoption_gate',
]].sort_values('mrr', ascending=False).style.format(precision=4))
display(pd.read_csv(OUT / 'geometry_diagnostics.csv').style.format(precision=4))
display(pd.read_csv(OUT / 'backend_subgroup_metrics.csv').style.format(precision=4))

In [ ]:
report = json.loads((OUT / 'backend_metrics.json').read_text())
print('DEV-SELECTED FAMILY:', report['dev_selected_family'])
print('RECOMMENDED BACKEND:', report['recommended_backend'])
print('PRODUCTION CHANGE AUTHORIZED:', report['production_change_authorized'])
print('RETURN:', OUT / 'backend_metrics.json')
print('RETURN:', OUT / 'backend_test_metrics.csv')
print('RETURN:', OUT / 'backend_author_exposure.csv')
print('RETURN:', OUT / 'backend_subgroup_metrics.csv')
print('RETURN:', OUT / 'geometry_diagnostics.csv')

## Part 2 · Concentration-constrained search

This continuation reuses the frozen embeddings and does not run the encoder. It searches raw–whitened blends, author-balanced whitening, whitening + CSLS, and author-balanced whitening + CSLS. Selection happens on dev only: minimize relative false-top3 HHI, Gini, and maximum share subject to MRR, Recall@3, and worst-decile Recall@3 non-inferiority. The already-opened test split is diagnostic, not confirmatory. No author receives a name-specific penalty.

In [ ]:
# Self-contained Part 2 setup: safe to start here in a fresh Colab runtime.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
GUTENBERG = REPO / 'artifacts/source_expansion_v2/gutenberg_targeted_v1'
HELDOUT = GUTENBERG / 'source_heldout_splits.parquet'
EMBEDDINGS = GUTENBERG / 'frozen_encoder_eval'
CONCENTRATION_OUT = GUTENBERG / 'concentration_search_v2'
assert HELDOUT.exists(), HELDOUT
for filename in ('style_embedding_train_embeddings.npy', 'style_embedding_train_chunk_ids.npy', 'style_embedding_eval_embeddings.npy', 'style_embedding_eval_chunk_ids.npy'):
    assert (EMBEDDINGS / filename).exists(), f'Missing reusable Part 8 embedding: {EMBEDDINGS / filename}'
def run(command):
    print('>>>', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {command}')

In [ ]:
# CPU-only search; no Part 1 command and no embedding batches are called.
run([
    sys.executable, 'scripts/search_concentration_backends.py',
    '--input', str(HELDOUT),
    '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(CONCENTRATION_OUT),
    '--train-cap', '300', '--bootstrap-runs', '5000',
    '--seed', '20260903',
    '--watch-profile', 'en::James Joyce',
    '--watch-profile', 'en::Katherine Mansfield',
    '--watch-profile', 'en::D. H. Lawrence',
])

In [ ]:
search = pd.read_csv(CONCENTRATION_OUT / 'concentration_dev_search.csv')
display(search.head(20)[[
    'candidate', 'mrr', 'recall_at_3', 'worst_decile_profile_recall_at_3',
    'false_top3_hhi', 'false_top3_gini', 'maximum_false_top3_share',
    'concentration_index', 'dev_eligible',
]].style.format(precision=4))
display(pd.read_csv(CONCENTRATION_OUT / 'concentration_test_diagnostic.csv').style.format(precision=4))
display(pd.read_csv(CONCENTRATION_OUT / 'concentration_watched_profiles.csv').style.format(precision=4))
selection = json.loads((CONCENTRATION_OUT / 'concentration_selection.json').read_text())
print(json.dumps(selection, indent=2))

In [ ]:
for filename in (
    'concentration_selection.json', 'concentration_dev_search.csv',
    'concentration_test_diagnostic.csv', 'concentration_author_exposure.csv',
    'concentration_watched_profiles.csv', 'concentration_subgroup_metrics.csv',
):
    print('RETURN:', CONCENTRATION_OUT / filename)

## Part 3 · Post-whitening hub calibration

Keeps `whitened_cosine:0.3` fixed and tests only post-whitening corrections: robust candidate-null scores, a cross-set mutual-proximity approximation, f-norm, exposure-aware subtraction, and a one-shot structural expert with reverse-rank anchors. Dev selects; the previously opened test split remains diagnostic. Named authors are watched but never receive name-specific weights.

In [ ]:
# Part 3A · Self-contained setup. Start here in a fresh Colab runtime.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib.util, json, os, subprocess, sys
missing = [name for name in ('pyarrow', 'sklearn', 'scipy') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow>=14', 'scikit-learn>=1.4', 'scipy>=1.11'], check=True)
import pandas as pd
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
GUTENBERG = REPO / 'artifacts/source_expansion_v2/gutenberg_targeted_v1'
HELDOUT = GUTENBERG / 'source_heldout_splits.parquet'
EMBEDDINGS = GUTENBERG / 'frozen_encoder_eval'
POSTWHITENING_OUT = GUTENBERG / 'postwhitening_calibration_v1'
assert HELDOUT.exists(), HELDOUT
for filename in ('style_embedding_train_embeddings.npy', 'style_embedding_train_chunk_ids.npy', 'style_embedding_eval_embeddings.npy', 'style_embedding_eval_chunk_ids.npy'):
    assert (EMBEDDINGS / filename).exists(), f'Missing reusable embedding: {EMBEDDINGS / filename}'
def run(command):
    print('>>>', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {command}')

In [ ]:
# Part 3B · CPU-only search on top of fixed whitened_cosine:0.3.
run([
    sys.executable, 'scripts/search_postwhitening_calibrators.py',
    '--input', str(HELDOUT),
    '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(POSTWHITENING_OUT),
    '--shrinkage', '0.3', '--train-cap', '300',
    '--bootstrap-runs', '5000', '--seed', '20260904',
    '--watch-profile', 'en::James Joyce',
    '--watch-profile', 'en::Katherine Mansfield',
    '--watch-profile', 'en::D. H. Lawrence',
])

In [ ]:
# Part 3C · Read the dev decision and any one-shot test diagnostic.
dev_search = pd.read_csv(POSTWHITENING_OUT / 'postwhitening_dev_search.csv')
display(dev_search.head(25)[[
    'candidate', 'mrr', 'recall_at_3', 'worst_decile_profile_recall_at_3',
    'source_balanced_maui_at_3', 'false_top3_hhi', 'false_top3_gini',
    'maximum_false_top3_share', 'concentration_index_vs_whitened',
    'subgroup_non_degradation', 'dev_eligible',
]].style.format(precision=4))
display(pd.read_csv(POSTWHITENING_OUT / 'postwhitening_dev_watched_profiles.csv').style.format(precision=4))
selection = json.loads((POSTWHITENING_OUT / 'postwhitening_selection.json').read_text())
print(json.dumps(selection, indent=2))
test_path = POSTWHITENING_OUT / 'postwhitening_test_diagnostic.csv'
if test_path.exists():
    display(pd.read_csv(test_path).style.format(precision=4))
    display(pd.read_csv(POSTWHITENING_OUT / 'postwhitening_watched_profiles.csv').style.format(precision=4))
geometry = pd.read_csv(POSTWHITENING_OUT / 'postwhitening_profile_geometry.csv')
display(geometry[geometry['profile'].isin(['en::James Joyce', 'en::Katherine Mansfield', 'en::D. H. Lawrence'])].style.format(precision=4))

In [ ]:
# Part 3D · Files to return.
for filename in (
    'postwhitening_selection.json', 'postwhitening_dev_search.csv',
    'postwhitening_dev_watched_profiles.csv',
    'postwhitening_profile_geometry.csv', 'postwhitening_test_diagnostic.csv',
    'postwhitening_author_exposure.csv', 'postwhitening_watched_profiles.csv',
    'postwhitening_subgroup_metrics.csv',
):
    path = POSTWHITENING_OUT / filename
    if path.exists():
        print('RETURN:', path)